# ПЗ 2 — Нарезка видео на кадры

**Задача:** скачать видео по URL в Google Drive, нарезать на кадры, сохранить в Drive.

**Порядок запуска:**
1. Запустить все ячейки сверху вниз
2. Вставить ссылку на видео в ячейку с `VIDEO_URL`
3. Кадры сохранятся в `MyDrive/cv-frames/кадры/`

**Стек:** `opencv-python`, `yt-dlp`, `tqdm`

In [ ]:
!pip install opencv-python-headless yt-dlp tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Drive смонтирован')
print(f'Видео:      {VIDEO_DIR}')
print(f'Кадры:      {FRAMES_DIR}')
print(f'Результаты: {RESULTS_DIR}')

In [ ]:
# === Вставьте ссылку на видео (YouTube, VK, Rutube и др.) ===
VIDEO_URL = 'https://www.youtube.com/watch?v=ВСТАВЬТЕ_ССЫЛКУ'

video_path = f'{VIDEO_DIR}/video.mp4'

!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" \
    -o "{video_path}" \
    --no-playlist \
    "{VIDEO_URL}"

if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / 1024 / 1024
    print(f'✅ Скачано: {video_path} ({size_mb:.1f} MB)')
else:
    print('❌ Не скачалось — проверьте ссылку')

In [ ]:
# Альтернатива: загрузить файл вручную
# from google.colab import files
# import shutil
# uploaded = files.upload()
# name = list(uploaded.keys())[0]
# video_path = f'{VIDEO_DIR}/{name}'
# shutil.move(name, video_path)
# print(f'✅ Загружено: {video_path}')

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print('❌ Видео не открылось — проверьте файл')
else:
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'✅ FPS: {fps:.1f} | Кадров: {total} | Длительность: {total/fps:.1f} сек')

In [ ]:
from tqdm.notebook import tqdm

FRAME_STEP = 30  # сохранять каждый N-й кадр

frame_idx = 0
saved = 0

for _ in tqdm(range(total), desc='Нарезка'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1

cap.release()
print(f'✅ Сохранено кадров: {saved} → {FRAMES_DIR}')

In [ ]:
# Показать первые 3 кадра
from IPython.display import Image, display

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))[:3]
for f in frames:
    print(f)
    display(Image(filename=f'{FRAMES_DIR}/{f}', width=400))